# ML exploration: WR next-week PPR (nflverse / ffverse)

Experiments for **next-week PPR** projections using `nflreadpy` + project modules under `backend/wr_predictor`.

In [1]:
# Run this cell first. If you see "Kernel ready" below, the kernel is working.
print("Kernel ready")

Kernel ready


In [2]:
# Project root on sys.path (same pattern as 01_data_exploration)
import os
import sys
from pathlib import Path

# wr_predictor now lives under backend/, so PROJECT_ROOT points at backend/ itself.
cwd = Path(os.getcwd())
REPO_ROOT = cwd if (cwd / "backend").exists() else cwd.parent
PROJECT_ROOT = REPO_ROOT / "backend"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root on sys.path:", PROJECT_ROOT)

Project root on sys.path: C:\Users\delga\Desktop\Personal\fantasy-web-app\backend


## 1. Load WR-week data

`build_training_dataset` loads nflverse weekly stats, merges **schedule** context (`spread_line`, `total_line`, `is_home`, `is_dome`, …), builds **lags / rolling** usage (`features.py`), optionally joins **ffopportunity** columns (`merge_ff_opportunity=True`), and adds `next_week_ppr_points`.


In [3]:
import polars as pl

from wr_predictor.dataset_builder import build_training_dataset

TRAINING_SEASONS = list(range(2015, 2025))  # adjust if your nflreadpy release lacks early years

wr_weekly = build_training_dataset(
    seasons=TRAINING_SEASONS,
    min_games_for_player=0,
    merge_ff_opportunity=True,
)

print(wr_weekly.shape)
wr_weekly.head()

(22053, 48)


player_id,player_name,player_display_name,season,week,season_type,team,opponent_team,is_home,is_dome,spread_line,total_line,temp,wind,ppr_points,ppr_points_prev_week,ppr_points_minus_roll3,rec_prev_week,targets_prev_week,rec_yds_prev_week,rec_td_prev_week,target_share_prev_week,air_yards_share_prev_week,wopr_prev_week,receiving_air_yards_prev_week,ppr_points_rolling_3,ppr_points_rolling_5,rec_yds_rolling_3,rec_yds_rolling_5,targets_rolling_3,targets_rolling_5,target_share_rolling_3,target_share_rolling_5,air_yards_share_rolling_3,air_yards_share_rolling_5,wopr_rolling_3,wopr_rolling_5,receiving_air_yards_rolling_3,receiving_air_yards_rolling_5,pass_fantasy_points_exp,rec_fantasy_points_exp,rush_fantasy_points_exp,pass_fantasy_points,rec_fantasy_points,rush_fantasy_points,total_fantasy_points,total_fantasy_points_exp,next_week_ppr_points
str,str,str,i32,i32,str,str,str,i8,i8,f64,f64,i32,i32,f64,f64,f64,i32,i32,i32,i32,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,1,"""REG""","""BAL""","""DEN""",0,0,4.5,46.5,88,13,3.3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,9.99,0.0,0.0,3.3,0.0,3.3,9.99,25.0
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,2,"""REG""","""BAL""","""LV""",0,0,-6.0,42.0,80,8,25.0,3.3,0.0,2,7,13,0,0.21875,0.3125,0.546875,60,3.3,3.3,13.0,13.0,7.0,7.0,0.21875,0.21875,0.3125,0.3125,0.546875,0.546875,60.0,60.0,0.0,28.27,0.0,0.0,25.0,0.0,25.0,28.27,43.6
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,3,"""REG""","""BAL""","""CIN""",1,0,2.5,45.5,70,12,43.6,25.0,10.85,10,16,150,0,0.355556,0.460177,0.855457,208,14.15,14.15,81.5,81.5,11.5,11.5,0.287153,0.287153,0.386338,0.386338,0.701166,0.701166,134.0,134.0,0.0,26.47,0.0,0.0,43.6,0.0,43.6,26.47,6.4
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,4,"""REG""","""BAL""","""PIT""",0,0,-3.0,44.0,63,8,6.4,43.6,19.633333,13,17,186,2,0.354167,0.298611,0.740278,86,23.966667,23.966667,116.333333,116.333333,13.333333,13.333333,0.309491,0.309491,0.357096,0.357096,0.714203,0.714203,118.0,118.0,0.0,9.74,0.0,0.0,6.4,0.0,6.4,9.74,26.7
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,6,"""REG""","""BAL""","""SF""",0,0,-2.5,43.5,69,7,26.7,6.4,-18.6,4,7,24,0,0.212121,0.187817,0.449654,37,25.0,19.575,120.0,93.25,13.333333,11.75,0.307281,0.285148,0.315535,0.314776,0.681796,0.648066,110.333333,97.75,0.0,20.02,0.0,0.0,26.7,0.0,26.7,20.02,12.8


## 2. Target and leakage

- **Target**: `next_week_ppr_points` — same-week `ppr_points` shifted **-1** within each `player_id` (sorted by season, week).
- **Features**: lags use `shift(1)`; rolling means apply `shift(1)` before the window so the **current game’s box score is not inside the rolling window** for that row.
- **Never** use the target or any future-week stats as inputs.


In [4]:
target_col = "next_week_ppr_points"

id_cols = [
    c
    for c in [
        "player_id",
        "player_name",
        "player_display_name",
        "season",
        "week",
        "team",
        "opponent_team",
        "home_away",
    ]
    if c in wr_weekly.columns
]

feature_cols = [c for c in wr_weekly.columns if c not in id_cols and c != target_col]
model_frame = wr_weekly.select(id_cols + feature_cols + [target_col])
model_frame.head()

player_id,player_name,player_display_name,season,week,team,opponent_team,season_type,is_home,is_dome,spread_line,total_line,temp,wind,ppr_points,ppr_points_prev_week,ppr_points_minus_roll3,rec_prev_week,targets_prev_week,rec_yds_prev_week,rec_td_prev_week,target_share_prev_week,air_yards_share_prev_week,wopr_prev_week,receiving_air_yards_prev_week,ppr_points_rolling_3,ppr_points_rolling_5,rec_yds_rolling_3,rec_yds_rolling_5,targets_rolling_3,targets_rolling_5,target_share_rolling_3,target_share_rolling_5,air_yards_share_rolling_3,air_yards_share_rolling_5,wopr_rolling_3,wopr_rolling_5,receiving_air_yards_rolling_3,receiving_air_yards_rolling_5,pass_fantasy_points_exp,rec_fantasy_points_exp,rush_fantasy_points_exp,pass_fantasy_points,rec_fantasy_points,rush_fantasy_points,total_fantasy_points,total_fantasy_points_exp,next_week_ppr_points
str,str,str,i32,i32,str,str,str,i8,i8,f64,f64,i32,i32,f64,f64,f64,i32,i32,i32,i32,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,1,"""BAL""","""DEN""","""REG""",0,0,4.5,46.5,88,13,3.3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,9.99,0.0,0.0,3.3,0.0,3.3,9.99,25.0
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,2,"""BAL""","""LV""","""REG""",0,0,-6.0,42.0,80,8,25.0,3.3,0.0,2,7,13,0,0.21875,0.3125,0.546875,60,3.3,3.3,13.0,13.0,7.0,7.0,0.21875,0.21875,0.3125,0.3125,0.546875,0.546875,60.0,60.0,0.0,28.27,0.0,0.0,25.0,0.0,25.0,28.27,43.6
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,3,"""BAL""","""CIN""","""REG""",1,0,2.5,45.5,70,12,43.6,25.0,10.85,10,16,150,0,0.355556,0.460177,0.855457,208,14.15,14.15,81.5,81.5,11.5,11.5,0.287153,0.287153,0.386338,0.386338,0.701166,0.701166,134.0,134.0,0.0,26.47,0.0,0.0,43.6,0.0,43.6,26.47,6.4
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,4,"""BAL""","""PIT""","""REG""",0,0,-3.0,44.0,63,8,6.4,43.6,19.633333,13,17,186,2,0.354167,0.298611,0.740278,86,23.966667,23.966667,116.333333,116.333333,13.333333,13.333333,0.309491,0.309491,0.357096,0.357096,0.714203,0.714203,118.0,118.0,0.0,9.74,0.0,0.0,6.4,0.0,6.4,9.74,26.7
"""00-0020337""","""S.Smith""","""Steve Smith""",2015,6,"""BAL""","""SF""","""REG""",0,0,-2.5,43.5,69,7,26.7,6.4,-18.6,4,7,24,0,0.212121,0.187817,0.449654,37,25.0,19.575,120.0,93.25,13.333333,11.75,0.307281,0.285148,0.315535,0.314776,0.681796,0.648066,110.333333,97.75,0.0,20.02,0.0,0.0,26.7,0.0,26.7,20.02,12.8


In [5]:
# Require non-null rolling-3 features; drop nulls in numeric feature/target columns


def _polars_numeric(schema, col: str) -> bool:
    dt = schema.get(col)
    return bool(dt) and getattr(dt, "is_numeric", lambda: False)()


numeric_cols = [c for c in feature_cols + [target_col] if _polars_numeric(wr_weekly.schema, c)]

roll3 = [c for c in model_frame.columns if c.endswith("_rolling_3")]
if roll3:
    mask = model_frame[roll3[0]].is_not_null()
    for c in roll3[1:]:
        mask = mask & model_frame[c].is_not_null()
    model_frame_clean = model_frame.filter(mask)
else:
    model_frame_clean = model_frame

model_frame_clean = model_frame_clean.drop_nulls(
    subset=[c for c in numeric_cols if c in model_frame_clean.columns]
)
model_frame_clean.shape

(13454, 48)

## 3. Time-aware train / validation / test split

- **Train**: seasons 2015–2021  
- **Validation**: 2022  
- **Test**: 2023–2024  


In [6]:
import pandas as pd

pdf = model_frame_clean.to_pandas()
season_col = "season"

train_mask = pdf[season_col].between(2015, 2021)
val_mask = pdf[season_col] == 2022
test_mask = pdf[season_col].between(2023, 2024)

X_cols = [
    c
    for c in feature_cols
    if c in pdf.columns and pd.api.types.is_numeric_dtype(pdf[c])
]
y_col = target_col

X_train, y_train = pdf.loc[train_mask, X_cols], pdf.loc[train_mask, y_col]
X_val, y_val = pdf.loc[val_mask, X_cols], pdf.loc[val_mask, y_col]
X_test, y_test = pdf.loc[test_mask, X_cols], pdf.loc[test_mask, y_col]

X_train.shape, X_val.shape, X_test.shape

ModuleNotFoundError: No module named 'pandas'

## 4. Ridge baseline

Grid over `alpha` using **validation RMSE** (time-ordered val year).


In [7]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:

    def root_mean_squared_error(y_true, y_pred):  # type: ignore[misc]
        return float(mean_squared_error(y_true, y_pred, squared=False) ** 0.5)


alphas = [0.1, 1.0, 10.0]
best_alpha = best_val_rmse = best_model = None

for alpha in alphas:
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha, random_state=0)),
        ]
    )
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_rmse = root_mean_squared_error(y_val, val_pred)
    if best_val_rmse is None or val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_alpha = alpha
        best_model = model

print(f"Best alpha: {best_alpha}, validation RMSE: {best_val_rmse:.3f}")

train_pred = best_model.predict(X_train)
val_pred = best_model.predict(X_val)
test_pred = best_model.predict(X_test)

for split_name, y_true, y_hat in [
    ("Train", y_train, train_pred),
    ("Validation", y_val, val_pred),
    ("Test", y_test, test_pred),
]:
    mae = mean_absolute_error(y_true, y_hat)
    rmse = root_mean_squared_error(y_true, y_hat)
    r2 = r2_score(y_true, y_hat)
    print(f"{split_name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R^2={r2:.3f}")

NameError: name 'X_train' is not defined

In [8]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual next-week PPR")
plt.ylabel("Predicted (Ridge)")
plt.title("Ridge: predicted vs actual (test)")
plt.tight_layout()
plt.show()

NameError: name 'y_test' is not defined

<Figure size 600x600 with 0 Axes>

## 5. Neural net baseline (`MLPRegressor`)

Same numeric `X_*` as Ridge. Uses **early stopping** on a holdout fraction of the training split (not the same as the 2022 validation year). For strict temporal validation of the NN, you could instead implement Keras/PyTorch with `X_val`/`y_val`.


In [9]:
from sklearn.neural_network import MLPRegressor

mlp_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                batch_size=256,
                max_iter=200,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=0,
            ),
        ),
    ]
)

mlp_model.fit(X_train, y_train)
mlp_test_pred = mlp_model.predict(X_test)

print(
    "MLP Test:",
    f"MAE={mean_absolute_error(y_test, mlp_test_pred):.3f}",
    f"RMSE={root_mean_squared_error(y_test, mlp_test_pred):.3f}",
    f"R^2={r2_score(y_test, mlp_test_pred):.3f}",
)

NameError: name 'X_train' is not defined

## 6. Interpretability

**Ridge**: standardized coefficient magnitudes.  
**MLP**: quick **permutation importance** on a small test slice (slow if you increase `n_repeats` / sample size).


In [10]:
ridge = best_model.named_steps["ridge"]
coefs = pd.Series(ridge.coef_, index=X_cols).abs().sort_values(ascending=False)
coefs.head(20)

AttributeError: 'NoneType' object has no attribute 'named_steps'

In [11]:
from sklearn.inspection import permutation_importance

sample_n = min(800, len(X_test))
X_perm = X_test.sample(sample_n, random_state=0)
y_perm = y_test.loc[X_perm.index]

r = permutation_importance(
    mlp_model,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=0,
    scoring="r2",
)
imp = pd.Series(r.importances_mean, index=X_cols).sort_values(ascending=False)
imp.head(15)

NameError: name 'X_test' is not defined

## 7. ffverse / ffopportunity note

With `merge_ff_opportunity=True`, numeric opportunity columns from `data_loader.load_ff_opportunity` are left-joined on `(player_id, season, week)` and kept in the modeling frame when present. If the join fails (missing keys / empty release), the notebook still runs with usage-only features.


## 8. Projections for a target week

For each WR, take the **last REG game strictly before** `(TARGET_SEASON, TARGET_WEEK)`, require the same non-null `_rolling_3` columns as training, then run `best_model.predict`.


In [12]:
TARGET_SEASON = 2024
TARGET_WEEK = 6


def projection_rows(df: pl.DataFrame) -> pl.DataFrame:
    prior = df.filter(
        (pl.col("season") < TARGET_SEASON)
        | ((pl.col("season") == TARGET_SEASON) & (pl.col("week") < TARGET_WEEK))
    )
    if "season_type" in prior.columns:
        prior = prior.filter(pl.col("season_type") == "REG")
    prior = prior.sort(["player_id", "season", "week"])
    return prior.unique(subset=["player_id"], keep="last")


proj_pl = projection_rows(wr_weekly)
if roll3:
    m = proj_pl[roll3[0]].is_not_null()
    for c in roll3[1:]:
        m = m & proj_pl[c].is_not_null()
    proj_pl = proj_pl.filter(m)

proj_pdf = proj_pl.to_pandas()
X_proj = proj_pdf[X_cols].astype(float)
proj_pdf = proj_pdf.assign(pred_next_week_ppr_ridge=best_model.predict(X_proj))

display_cols = [c for c in ("player_display_name", "player_name", "team", "season", "week") if c in proj_pdf.columns]
proj_pdf.sort_values("pred_next_week_ppr_ridge", ascending=False)[
    display_cols + ["pred_next_week_ppr_ridge"]
].head(25)

ModuleNotFoundError: No module named 'pyarrow'

## 9. Summary and next steps

- **Data**: nflverse via `nflreadpy`; schedules for game context; optional ffopportunity via `merge_ff_opportunity`.
- **Target**: `next_week_ppr_points`.
- **Models**: Ridge (strong linear baseline) + `MLPRegressor`.
- **Next**: opponent defensive metrics, injury/participation filters, gradient boosting (`HistGradientBoostingRegressor`, XGBoost), calibration / quantile models for uncertainty.


## 10. 2025 backtest: boom/bust residual analysis

Train on **2021-2024**, predict on the held-out **2025** season (a season the model never saw), then look at the player-weeks where the model was most wrong in each direction:
- **Boom misses**: actual >> predicted (model missed a breakout/spike performance).
- **Bust misses**: actual << predicted (model expected more than the player produced).

Goal: use these outliers to spot situational factors the current feature set doesn't capture (e.g. injuries elsewhere in the offense, blowout game script, defensive matchup strength) before trusting this model on live 2026 projections.

In [13]:
from wr_predictor.dataset_builder import build_training_dataset
from wr_predictor.model import get_feature_columns, prepare_model_frame, split_by_season

BACKTEST_SEASONS = [2021, 2022, 2023, 2024, 2025]
backtest_weekly = build_training_dataset(seasons=BACKTEST_SEASONS, min_games_for_player=0)

# Drop postseason rows as train/validation examples (regular-season forecasting only),
# but do this AFTER lag/rolling features were computed above, so a player's playoff
# games still inform their rolling trend going into next season's week 1 (per Sept 2026
# design discussion: postseason results shouldn't count as data points, but the trend
# they represent is still real signal).
if "season_type" in backtest_weekly.columns:
    before = backtest_weekly.height
    backtest_weekly = backtest_weekly.filter(pl.col("season_type") == "REG")
    print(f"Dropped {before - backtest_weekly.height} postseason rows.")

backtest_feature_cols = get_feature_columns(backtest_weekly)
backtest_frame = prepare_model_frame(backtest_weekly, backtest_feature_cols)
backtest_train, backtest_validation = split_by_season(
    backtest_frame, train_seasons=[2021, 2022, 2023, 2024], validation_seasons=[2025]
)
print(f"Train rows: {backtest_train.height}, 2025 validation rows: {backtest_validation.height}")

Dropped 457 postseason rows.
Dropped 4937 of 11054 rows with missing feature/target values.
Train rows: 4785, 2025 validation rows: 1332


In [14]:
from sklearn.linear_model import Ridge

x_train = backtest_train.select(backtest_feature_cols).to_numpy()
y_train = backtest_train["next_week_ppr_points"].to_numpy()
x_2025 = backtest_validation.select(backtest_feature_cols).to_numpy()
y_2025_actual = backtest_validation["next_week_ppr_points"].to_numpy()

backtest_model = Ridge(alpha=1.0)
backtest_model.fit(x_train, y_train)
y_2025_predicted = backtest_model.predict(x_2025)

context_cols = [
    c
    for c in [
        "player_name",
        "player_display_name",
        "season",
        "week",
        "team",
        "opponent_team",
        "home_away",
        "is_home",
        "is_dome",
        "spread_line",
        "total_line",
        "temp",
        "wind",
    ]
    if c in backtest_validation.columns
]

residuals_df = (
    backtest_validation.select(context_cols)
    .with_columns(
        [
            pl.Series("actual_ppr", y_2025_actual),
            pl.Series("predicted_ppr", y_2025_predicted),
        ]
    )
    .with_columns((pl.col("actual_ppr") - pl.col("predicted_ppr")).alias("residual"))
)
print(f"MAE 2025: {abs(residuals_df['residual']).mean():.3f}")

MAE 2025: 4.675


In [15]:
top_boom = residuals_df.sort("residual", descending=True).head(15)
top_bust = residuals_df.sort("residual").head(15)

print("Top BOOM misses (actual far ABOVE predicted):")
print(top_boom)
print()
print("Top BUST misses (actual far BELOW predicted):")
print(top_bust)

Top BOOM misses (actual far ABOVE predicted):
shape: (15, 15)
┌─────────────┬────────────────┬────────┬──────┬───┬──────┬────────────┬───────────────┬───────────┐
│ player_name ┆ player_display ┆ season ┆ week ┆ … ┆ wind ┆ actual_ppr ┆ predicted_ppr ┆ residual  │
│ ---         ┆ _name          ┆ ---    ┆ ---  ┆   ┆ ---  ┆ ---        ┆ ---           ┆ ---       │
│ str         ┆ ---            ┆ i32    ┆ i32  ┆   ┆ i32  ┆ f64        ┆ f64           ┆ f64       │
│             ┆ str            ┆        ┆      ┆   ┆      ┆            ┆               ┆           │
╞═════════════╪════════════════╪════════╪══════╪═══╪══════╪════════════╪═══════════════╪═══════════╡
│ Mi.Wilson   ┆ Michael Wilson ┆ 2025   ┆ 10   ┆ … ┆ 5    ┆ 33.5       ┆ 7.438522      ┆ 26.061478 │
│ D.London    ┆ Drake London   ┆ 2025   ┆ 7    ┆ … ┆ 5    ┆ 38.8       ┆ 15.163998     ┆ 23.636002 │
│ M.Wilson    ┆ Michael Wilson ┆ 2025   ┆ 13   ┆ … ┆ 8    ┆ 37.2       ┆ 14.071136     ┆ 23.128864 │
│ R.Odunze    ┆ Rome Odunze  

### Notes: what drove these misses?

_Fill in after reviewing the tables above — e.g. injuries to a teammate that shifted target share, a blowout that changed game script, a tough/weak defensive matchup not in the current feature set, weather, or a quarterback change. These observations are candidate features for a future model iteration._